In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [7]:
TICKERS = {
    "Australia": "SAUS.L",
    "Brazil":    "XMBR.L",
    "Canada":    "CSCA.L",
    "China":     "IASH.L",
    "France":    "ISFR.L",
    "Germany":   "XDAX.L",
    "India":     "IIND.L",
    "Indonesia": "HIDRI.L",
    "Italy":     "CMIB.L",
    "Japan":     "LCJP.L",
    "Poland":    "SPOL.L",
}


end   = datetime.today().strftime("%Y-%m-%d")
start = (datetime.today() - timedelta(days=365 * 10)).strftime("%Y-%m-%d")

In [8]:
# ── Download ──────────────────────────────────────────────────────────────────
raw = yf.download(list(TICKERS.values()), start=start, end=end, auto_adjust=True)["Close"]
raw.columns = [next(k for k, v in TICKERS.items() if v == col) for col in raw.columns]
raw.index.name = "Date"

countries = list(raw.columns)

[**********************82%**************         ]  9 of 11 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HIDRI.L"}}}
$HIDRI.L: possibly delisted; no timezone found
[*********************100%***********************]  10 of 11 completed

1 Failed download:
['HIDRI.L']: possibly delisted; no timezone found


In [9]:
countries

['Italy',
 'Canada',
 'Indonesia',
 'China',
 'India',
 'France',
 'Japan',
 'Australia',
 'Poland',
 'Germany',
 'Brazil']

In [10]:
# ── Derived columns per country ───────────────────────────────────────────────
#
#  _price        : adjusted closing price
#  _daily_pct    : day-over-day % change        → daily performance
#  _cumul_pct    : cumulative % return vs day 1 → total growth from start
#  _index        : base-100 price index         → CAGR = (end_index/100)^(1/years) - 1
#  _log_return   : cumulative log return        → additive, useful for return decomposition
#
frames = []

for country in countries:
    prices = raw[country]
    base   = prices.iloc[0]

    col = pd.DataFrame(index=raw.index)
    col[f"{country}_price"]       = prices
    col[f"{country}_daily_pct"]   = prices.pct_change()
    col[f"{country}_cumul_pct"]   = (prices / base) - 1
    col[f"{country}_index"]       = (prices / base) * 100
    col[f"{country}_log_return"]  = np.log(prices / base)
    frames.append(col)

df = pd.concat(frames, axis=1).reset_index()

# Year + Month columns make pivot tables and groupby easy in Excel
df.insert(1, "Year",  pd.to_datetime(df["Date"]).dt.year)
df.insert(2, "Month", pd.to_datetime(df["Date"]).dt.month)

print(df.head())
print(f"\nShape: {df.shape}  ({len(countries)} countries × 5 metrics + date cols)")



        Date  Year  Month  Italy_price  Italy_daily_pct  Italy_cumul_pct  \
0 2016-03-01  2016      3          NaN              NaN              NaN   
1 2016-03-02  2016      3          NaN              NaN              NaN   
2 2016-03-03  2016      3          NaN              NaN              NaN   
3 2016-03-04  2016      3          NaN              NaN              NaN   
4 2016-03-07  2016      3          NaN              NaN              NaN   

   Italy_index  Italy_log_return  Canada_price  Canada_daily_pct  ...  \
0          NaN               NaN        6818.5               NaN  ...   
1          NaN               NaN        6722.0         -0.014153  ...   
2          NaN               NaN        6776.0          0.008033  ...   
3          NaN               NaN        6847.5          0.010552  ...   
4          NaN               NaN        6939.5          0.013436  ...   

   Germany_price  Germany_daily_pct  Germany_cumul_pct  Germany_index  \
0    9540.969727               

In [11]:
df


,Date,Year,Month,Italy_price,Italy_daily_pct,Italy_cumul_pct,Italy_index,Italy_log_return,Canada_price,Canada_daily_pct,...,Germany_price,Germany_daily_pct,Germany_cumul_pct,Germany_index,Germany_log_return,Brazil_price,Brazil_daily_pct,Brazil_cumul_pct,Brazil_index,Brazil_log_return
0,2016-03-01,2016,3,NaN,NaN,NaN,NaN,NaN,6818.5,NaN,...,9540.969727,NaN,0.000000,100.000000,0.000000,1644.25,NaN,0.000000,100.000000,0.000000
1,2016-03-02,2016,3,NaN,NaN,NaN,NaN,NaN,6722.0,-0.014153,...,9599.330078,0.006117,0.006117,100.611682,0.006098,1683.75,0.024023,0.024023,102.402311,0.023739
2,2016-03-03,2016,3,NaN,NaN,NaN,NaN,NaN,6776.0,0.008033,...,9575.049805,-0.002529,0.003572,100.357197,0.003566,1731.25,0.028211,0.052912,105.291166,0.051559
3,2016-03-04,2016,3,NaN,NaN,NaN,NaN,NaN,6847.5,0.010552,...,9645.980469,0.007408,0.011006,101.100630,0.010946,1904.75,0.100217,0.158431,115.843090,0.147066
4,2016-03-07,2016,3,NaN,NaN,NaN,NaN,NaN,6939.5,0.013436,...,9601.480469,-0.004613,0.006342,100.634220,0.006322,1894.50,-0.005381,0.152197,115.219705,0.141671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2520,2026-02-20,2026,2,NaN,NaN,NaN,NaN,NaN,21788.5,0.001678,...,20650.000000,0.006581,1.164350,216.435023,0.772120,5255.50,0.004684,2.196290,319.629010,1.161991
2521,2026-02-23,2026,2,NaN,NaN,NaN,NaN,NaN,21734.0,-0.002501,...,20462.500000,-0.009080,1.144698,214.469814,0.762999,5266.50,0.002093,2.202980,320.298008,1.164082
2522,2026-02-24,2026,2,NaN,NaN,NaN,NaN,NaN,21710.0,-0.001104,...,20395.000000,-0.003299,1.137623,213.762338,0.759695,5306.00,0.007500,2.227003,322.700319,1.171554
2523,2026-02-25,2026,2,NaN,NaN,NaN,NaN,NaN,21971.0,0.012022,...,20555.000000,0.007845,1.154393,215.439317,0.767509,5329.50,0.004429,2.241295,324.129542,1.175973


In [12]:
df.to_csv("etf_prices.csv", index=False)
# df.to_excel("etf_prices.xlsx", index=False)  # uncomment for Excel
